In [ ]:
from polychrom.hdf5_format import list_URIs, load_URI
from polychrom import polymerutils
import numpy as np
import math
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


In [ ]:
folder_name = "/path/to/trajectory"
URIs = list_URIs(folder_name)
len(URIs)
data = [polymerutils.load(filename) for filename in URIs]

In [ ]:
N = 2800
dsb_site_right = 700
dsb_site_left = dsb_site_right - 1
contact_threshold = 5 

# Key positions in unbinned coordinates
DSB_time_unbinned = 100
homology_center_unbinned = 2100

## Helper Functions

In [ ]:
def find_homology_binding_time(contacts_time_map, homology_center):
    """
    Returns:
        t_bind (int or None): first time index where binding occurs,
                              or None if no stable binding
    """
    contact_trace = contacts_time_map[:, homology_center].astype(bool)
    T = len(contact_trace)

    for t in range(T):
        if contact_trace[t] and contact_trace[t:].all():
            return t

    return None

def distance_to_nearest_factor(factor_time_pos, t, site):
    """
    factor_time_pos: (T, N) binary array
    t: time index
    site: position index

    Returns:
        min distance (int), or np.inf if no factor present
    """
    positions = np.where(factor_time_pos[t] > 0)[0]
    if len(positions) == 0:
        return np.inf
    return np.min(np.abs(positions - site))

def build_contacts_map(
    data,
    instance,
    N,
    dsb_site_left,
    dsb_site_right,
    contact_threshold
):
    """
    Builds a (T, N) binary contact map where a contact is recorded
    if a monomer is within contact_threshold of EITHER DSB end.

    Works for both tethered and untethered cases.
    """
    offset = instance * N

    # Positions of both DSB ends over time
    dsb_left_pos = np.array([
        frame[offset + dsb_site_left] for frame in data
    ])
    dsb_right_pos = np.array([
        frame[offset + dsb_site_right] for frame in data
    ])

    contacts = np.zeros((len(data), N), dtype=np.uint8)

    for t, frame in enumerate(data):
        segment = frame[offset:offset + N]

        # Distances to each DSB end
        d_left = np.linalg.norm(segment - dsb_left_pos[t], axis=1)
        d_right = np.linalg.norm(segment - dsb_right_pos[t], axis=1)

        # Contact if close to EITHER end
        contacts[t] = (d_left <= contact_threshold) | (d_right <= contact_threshold)

    return contacts


def build_factor_map(positions, instance, N):
    T = len(positions)
    fmap = np.zeros((T, N), dtype=np.uint8)

    lo = instance * N
    hi = (instance + 1) * N

    for t, events in enumerate(positions):
        if len(events) == 0:
            continue

        events = np.asarray(events, dtype=np.int64)

        mask = (events[:, 0] >= lo) & (events[:, 0] < hi)
        if not np.any(mask):
            continue

        left = events[mask, 0] - lo
        right = events[mask, 1] - lo

        fmap[t, left] = 1
        fmap[t, right] = 1

    return fmap

def build_factor_contact_map(factor_time_pos, data, instance, N, spatial_threshold,
                              factor_stride=10):
    """
    factor_stride: ratio of factor_time_pos timesteps to data frames.
                   Set to 10 to match the 1-in-10 downsampling in the plotting cell.
    """
    T_data = len(data)
    T_factor = len(factor_time_pos)
    offset = instance * N
    contact_map = np.zeros((T_data, N), dtype=np.uint8)

    for t_data in range(T_data):
        t_factor = t_data * factor_stride          
        if t_factor >= T_factor:
            break

        factor_indices = np.where(factor_time_pos[t_factor] > 0)[0]
        if len(factor_indices) == 0:
            continue

        frame = data[t_data]                           # use data-space index here
        segment   = frame[offset:offset + N]           # (N, 3)
        factor_xyz = frame[offset + factor_indices]    # (n_factors, 3)

        # vectorized: (N, F) distance matrix
        dists = np.linalg.norm(
            segment[:, None, :] - factor_xyz[None, :, :], axis=2
        )
        contact_map[t_data] = (dists.min(axis=1) <= spatial_threshold).astype(np.uint8)

    return contact_map

def analyze_instance(
    contacts_time_map,
    cohesin_time_pos,
    cohesion_time_pos,
    homology_center,
    data,
    instance,
    N,
    spatial_threshold=10,   # matches contact_threshold
):
    t_bind = find_homology_binding_time(contacts_time_map, homology_center)
    mechanism = classify_binding_mechanism(
        t_bind=t_bind,
        homology_center=homology_center,
        cohesin_time_pos=cohesin_time_pos,
        cohesion_time_pos=cohesion_time_pos,
        data=data,
        instance=instance,
        N=N,
        spatial_threshold=spatial_threshold,
    )
    return {"t_bind": t_bind, "mechanism": mechanism}

def classify_binding_mechanism(
    t_bind,
    homology_center,
    cohesin_time_pos,
    cohesion_time_pos,
    data,
    instance,
    N,
    spatial_threshold,  
):
    """
    Returns one of {"LEF", "Cohesive", "Both", "Neither",
                    "Homology Not Found", "Homology Found Instantly"}

    Proximity is determined by 3D Euclidean distance <= spatial_threshold,
    matching the contact_threshold used in build_contacts_map.
    Index-space positions in cohesin_time_pos / cohesion_time_pos are
    untouched and remain available for plotting.
    """
    if t_bind is None:
        return "Homology Not Found"
    elif t_bind == 0:
        return "Homology Found Instantly"

    t_check = t_bind * 10 - 1  # adjust for contacts map downsampling

    d_lef = distance_to_nearest_factor_3d(
        cohesin_time_pos, data, instance, N, t_check, homology_center
    )
    d_cohesive = distance_to_nearest_factor_3d(
        cohesion_time_pos, data, instance, N, t_check, homology_center
    )

    lef_present      = d_lef      <= spatial_threshold
    cohesive_present = d_cohesive <= spatial_threshold

    if lef_present and cohesive_present:
        return "Both"
    elif lef_present:
        return "LEF"
    elif cohesive_present:
        return "Cohesive"
    else:
        return "Neither"

def normalize(x):
    x = x.astype(float)
    if x.max() > 0:
        x /= x.max()
    return x

def bin_2d(arr, time_bin, pos_bin, mode="max"):
    T, N = arr.shape
    T2 = T // time_bin
    N2 = N // pos_bin
    trimmed = arr[:T2 * time_bin, :N2 * pos_bin]
    binned = trimmed.reshape(T2, time_bin, N2, pos_bin)
    if mode == "sum":
        return binned.sum(axis=(1, 3))
    elif mode == "max":
        return binned.max(axis=(1, 3))
    elif mode == "mean":
        return binned.mean(axis=(1, 3))
    else:
        raise ValueError

def build_proximal_lef_map(LEF_Positions_all, data, instance, N,
                            t_bind, homology_center, spatial_threshold):
    T_data = len(data)
    offset = instance * N
    proximal_map = np.zeros((T_data, N), dtype=np.float32)

    if t_bind is None or t_bind == 0:
        return proximal_map

    t_data_bind   = max(0, t_bind - 1)
    t_factor_bind = min(t_data_bind * 10, len(LEF_Positions_all) - 1)

    lo = instance * N
    hi = (instance + 1) * N

    events = LEF_Positions_all[t_factor_bind]
    if len(events) == 0:
        return proximal_map
    events = np.asarray(events, dtype=np.int64)

    mask   = (events[:, 0] >= lo) & (events[:, 0] < hi)
    events = events[mask]
    if len(events) == 0:
        return proximal_map

    local_left  = events[:, 0] - lo
    local_right = events[:, 1] - lo

    frame    = data[t_data_bind]
    site_xyz = frame[offset + homology_center]

    proximal_lef_pairs = []
    for i in range(len(local_left)):
        ll, lr = local_left[i], local_right[i]
        d_left  = np.linalg.norm(frame[offset + ll] - site_xyz)
        d_right = np.linalg.norm(frame[offset + lr] - site_xyz)
        if min(d_left, d_right) <= spatial_threshold:
            proximal_lef_pairs.append((ll, lr))

    if not proximal_lef_pairs:
        return proximal_map

    def trace_lef(pair, t_factor_start, direction):
        ll_seed, lr_seed = pair
        t = t_factor_start
        while 0 <= t < len(LEF_Positions_all):
            evts = LEF_Positions_all[t]
            if len(evts) == 0:
                break
            evts = np.asarray(evts, dtype=np.int64)
            mask = (evts[:, 0] >= lo) & (evts[:, 0] < hi)
            evts = evts[mask]
            if len(evts) == 0:
                break
            loc_l = evts[:, 0] - lo
            loc_r = evts[:, 1] - lo
            diffs = np.abs(loc_l - ll_seed)
            closest = np.argmin(diffs)
            if diffs[closest] > 2:
                break
            ll_seed = loc_l[closest]
            lr_seed = loc_r[closest]
            yield t, ll_seed, lr_seed
            t += direction

    for pair in proximal_lef_pairs:
        factor_frames = {}
        for t_f, ll, lr in trace_lef(pair, t_factor_bind, direction=-1):
            factor_frames[t_f] = (ll, lr)
        for t_f, ll, lr in trace_lef(pair, t_factor_bind + 1, direction=+1):
            factor_frames[t_f] = (ll, lr)
        for t_f, (ll, lr) in factor_frames.items():
            t_d = t_f // 10
            if 0 <= t_d < T_data:
                proximal_map[t_d, ll] = 1.0
                proximal_map[t_d, lr] = 1.0

    return proximal_map

def format_time_axis(ax, dsb_time_binned, seconds_per_row):
    y_min, y_max = ax.get_ylim()
    sec_min = (y_min - dsb_time_binned) * seconds_per_row
    sec_max = (y_max - dsb_time_binned) * seconds_per_row
    tick_sec = np.arange(
        np.ceil(sec_min / 100) * 100,
        np.floor(sec_max / 100) * 100 + 1,
        100
    )
    tick_rows = tick_sec / seconds_per_row + dsb_time_binned
    ax.set_yticks(tick_rows)
    ax.set_yticklabels([f"{int(s)}" for s in tick_sec])

def format_position_axis_with_zero(ax, homology_center_plot):
    x_min, x_max = ax.get_xlim()
    kb_min = (x_min - homology_center_plot) * pos_bin
    kb_max = (x_max - homology_center_plot) * pos_bin
    tick_kb = np.arange(
        np.ceil(kb_min / 100) * 100,
        np.floor(kb_max / 100) * 100 + 1,
        100
    )
    tick_cols = tick_kb / pos_bin + homology_center_plot
    ax.set_xticks(tick_cols)
    ax.set_xticklabels([f"{int(k)}" for k in tick_kb])





## Single Trace Plot

In [ ]:
instance = 0

LEF_Positions_all = np.load(
    "/path/to/LEF_Positions.npy",
    allow_pickle=True
)

Cohesion_Positions_all = np.load(
    "/path/to/RecruitedCohesiveCohesin_Positions.npy",
    allow_pickle=True
)

contacts_time_map = build_contacts_map(
    data,
    instance,
    N,
    dsb_site_left,
    dsb_site_right,
    contact_threshold
)


cohesin_time_pos_raw = build_factor_map(
    LEF_Positions_all, instance, N
)

cohesion_time_pos_raw = build_factor_map(
    Cohesion_Positions_all, instance, N
)

cohesin_time_pos = np.zeros((len(data), N))
cohesion_time_pos = np.zeros((len(data), N))

for i in range(min(len(cohesin_time_pos_raw), len(data) * 10)):
    if i % 10 == 0:
        cohesin_time_pos[int(i/10)] = cohesin_time_pos_raw[i]
        cohesion_time_pos[int(i/10)] = cohesion_time_pos_raw[i]


In [ ]:
time_bin = 1
pos_bin = 15
aspect = 0.75

time_range_low = 100
time_range_high = 600
DSB_time_in_window = DSB_time_unbinned - time_range_low

x0 = 1400 // pos_bin
x1 = 2800 // pos_bin

analysis = analyze_instance(
    contacts_time_map,
    cohesin_time_pos_raw,
    cohesion_time_pos_raw,
    homology_center_unbinned,
    data=data,
    instance=instance,
    N=N,
    spatial_threshold=10,
)
t_bind    = analysis["t_bind"]
mechanism = analysis["mechanism"]

use_spatial_factor_maps = False

if use_spatial_factor_maps:
    cohesion_plot_map = build_factor_contact_map(
        cohesion_time_pos_raw, data, instance, N,
        spatial_threshold=contact_threshold
    )
    cohesin_plot_map = cohesin_time_pos
else:
    cohesion_plot_map = cohesion_time_pos
    cohesin_plot_map  = cohesin_time_pos

proximal_lef_map = build_proximal_lef_map(
    LEF_Positions_all, data, instance, N,
    t_bind=t_bind,
    homology_center=homology_center_unbinned,
    spatial_threshold=10,
)

R_b = bin_2d(contacts_time_map[time_range_low:time_range_high], time_bin, pos_bin)
G_b = bin_2d(cohesion_plot_map[time_range_low:time_range_high], time_bin, pos_bin)
B_b = bin_2d(cohesin_plot_map[time_range_low:time_range_high],  time_bin, pos_bin)
P_b = bin_2d(proximal_lef_map[time_range_low:time_range_high],  time_bin, pos_bin, mode="max")

# intensity / color
lef_base_intensity = 0.5
lef_proximal_boost = 0.5
contact_boost      = 1.5

R = np.clip(normalize(R_b) * contact_boost, 0, 1)
G = normalize(G_b)
B = normalize(B_b)
P = normalize(P_b)

B_display = np.clip(B * lef_base_intensity + P * lef_proximal_boost, 0, 1)

# Composite
background = np.ones_like(R)

R_signal = R * 0.6
G_signal = G * 0.9
B_signal = B_display * 0.9

rgb_white_bg = np.stack([
    background - G_signal - B_signal + R_signal,
    background - R_signal - B_signal + G_signal,
    background - R_signal - G_signal + B_signal,
], axis=-1)
rgb_white_bg = np.clip(rgb_white_bg, 0, 1)

# Coordinates
dsb_time_binned            = DSB_time_in_window // time_bin
homology_center_binned_abs = homology_center_unbinned // pos_bin
homology_center_plot       = homology_center_binned_abs - x0

AU_TO_SECONDS    = 2 / 3 * 10
seconds_per_row  = time_bin * AU_TO_SECONDS

# Plot
plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
})

#fig, axs = plt.subplots(1, 4, figsize=(14, 5), sharey=True)
fig, axs = plt.subplots(1, 4, figsize=(12, 7), sharey=True)

# Channels
axs[0].imshow(
    np.stack([R + 1, 1 - R, 1 - R], axis=-1)[:, x0:x1],
    origin="lower", aspect=aspect
)
axs[0].set_title("DSB-Sister Contacts")

axs[1].imshow(
    np.stack([1 - G, G + 1, 1 - G], axis=-1)[:, x0:x1],
    origin="lower", aspect=aspect
)
axs[1].set_title("Cohesive Clamps")

axs[2].imshow(
    np.stack([1 - B_display, 1 - B_display, B_display + 1], axis=-1)[:, x0:x1],
    origin="lower", aspect=aspect
)
axs[2].set_title("LEFs")

axs[3].imshow(
    rgb_white_bg[:, x0:x1],
    origin="lower",
    aspect=aspect,
    interpolation="none"
)
axs[3].set_title("Composite")

# Formatting
if False:
    for ax in axs:
        format_time_axis(ax, dsb_time_binned, seconds_per_row)
        format_position_axis_with_zero(ax, homology_center_plot)
        #ax.axvline(
        #    x=homology_center_plot,
        #    color='black',
        #    linestyle='--',
        #    linewidth=1,
        #    alpha=0.4
        #)

    # y-label only on left
    axs[0].set_ylabel("Time Since DSB (s)")
    for ax in axs[1:]:
        ax.set_ylabel("")

    # shared x label
    fig.supxlabel("Distance from homology (kb)", fontsize=18)

    plt.subplots_adjust(wspace=0.05)
plt.show()